In [26]:
pip install unstructured

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ------------------- ---------------- 524.3/981.5 kB 424.8 kB/s eta 0:00:02
     ------------------- ---------------- 524.3/981.5 kB 424.8 kB/s eta 0:00:02
     ------------------- ---------------- 524.3/981.5 kB 424.8 kB/s eta 0:00:02
     ------------------- ---------------- 524.3/981.5 kB 424.8 kB/s eta 0:00:02
     ---------------------------- ------- 786.4/981.5 kB 395.6 kB/s eta 0:00:01
     ---------------------------- ------- 786.4/981.5 kB 395.6 kB/s eta 0:00:01
     ------------------------------------ 981.5/981.5 kB 400.3 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.

  DEPRECATION: Building 'langdetect' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'langdetect'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [1]:
pip install python-pptx pypdf



   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   ---------------------------------------- 2/2 [python-pptx]

Note: you may need to restart the kernel to use updated packages.


## Taking All Data

In [3]:
import os
from pptx import Presentation
from pypdf import PdfReader

def extract_all_text(directory_path):
    all_content = ""
    
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        
        # Process PDF files
        if filename.lower().endswith(".pdf"):
            try:
                reader = PdfReader(file_path)
                pdf_text = ""
                for page in reader.pages:
                    pdf_text += page.extract_text() + "\n"
                all_content += f"\n--- Start PDF: {filename} ---\n{pdf_text}"
            except Exception as e:
                print(f"Could not read PDF {filename}: {e}")

        # Process PPTX files
        elif filename.lower().endswith(".pptx"):
            try:
                prs = Presentation(file_path)
                pptx_text = ""
                for slide in prs.slides:
                    for shape in slide.shapes:
                        if hasattr(shape, "text"):
                            pptx_text += shape.text + " "
                    pptx_text += "\n"
                all_content += f"\n--- Start PPTX: {filename} ---\n{pptx_text}"
            except Exception as e:
                print(f"Could not read PPTX {filename}: {e}")
                
    return all_content

# Usage
folder_path = "SQL Teacher/"
final_data = extract_all_text(folder_path)
# print(final_data)


## Use this

## Embeddings & Creating FAISS DB

In [18]:
pip install langchain-community faiss-cpu


In [20]:
from sentence_transformers import SentenceTransformer
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [21]:
embeddings_ollama = OllamaEmbeddings(model='mistral')

In [23]:
V_db = FAISS.from_documents(documents=final_data,embedding=embeddings_ollama)

AttributeError: 'str' object has no attribute 'page_content'

In [6]:
modell_sentence = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
embedding_data = modell_sentence.encode(final_data)

## Creating a DB

In [9]:
import faiss
import numpy as np

In [10]:
index = faiss.IndexFlatL2(embedding_data.shape[1])

IndexError: tuple index out of range

## Full Part Only Langchain

In [28]:
import os
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPowerPointLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load your documents
def load_docs(directory):
    docs = []
    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        if file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            docs.extend(loader.load())
        elif file.endswith(".pptx"):
            loader = UnstructuredPowerPointLoader(file_path)
            docs.extend(loader.load())
    return docs

documents = load_docs("SQL Teacher/")

In [29]:
# 2. Split text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

In [41]:
# 3. Create FAISS Vector Store (requires an API key for embeddings)
# Replace with HuggingFaceEmbeddings() for a free, local alternative
#here we initializing ollama emddings'
embeddings = OllamaEmbeddings(model = 'mistral')

vector_db = FAISS.from_documents(chunks, embeddings)     #here we pass chunk, embeddings -->inside/internally chunk converting into embedding .and chunk getting the index of embeddings.....these embeddings and chunks store into FAISS
#Example
#i          am        lucky
#[0.9701     0.8911    0.3890]

In [33]:
# 4. Search your data
query = "What is the main topic of my presentation?"
results = vector_db.similarity_search(query, k=3)

for res in results:
    print(f"Source: {res.metadata['source']}\nContent: {res.page_content[:200]}...")

Source: SQL Teacher/Data_Warehousing_Data_Mining_Big_Data.pptx
Content: Hash Table in Data Structure

A hash table is a data structure that efficiently stores and retrieves key-value pairs using a hash function to compute an index for each key. The key-value pairs are sto...
Source: SQL Teacher/Data_Warehousing_Data_Mining_Big_Data.pptx
Content: Local conceptual schema

Description of the database that reflect entities, data attributes, relationships and constraints.

User: Users in Europe, including their personal details.

Local internal sc...
Source: SQL Teacher/Data_Warehousing_Data_Mining_Big_Data.pptx
Content: The analysis layer in the architecture of the data warehouse features aggregate information navigators and efficient query optimizers.

The Two-tier architecture has a primary drawback is that it is n...


In [37]:
embeddings

OllamaEmbeddings(model='mistral', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [38]:
## Storing the vector database into file
# Saves the index, docstore, and index_to_docstore_id to a folder
vector_db.save_local("my_faiss_index") 


## Model

In [46]:
from langchain_ollama import ChatOllama

In [71]:
model = ChatOllama(model = "mistral")

## Prompt

In [72]:
from langchain_core.prompts import PromptTemplate
#condense question prompt
condense_question_prompt = PromptTemplate.from_template("""
given the following conversation and a follow up question, rephrase the follow up question to be standalone question
Chat History:
{chat_history}
Follow Up Input: {question}
standalone questions: """)

## Parser


In [73]:
from langchain_core.output_parsers import StrOutputParser

In [74]:
parser = StrOutputParser()

## Chain

In [75]:
from langchain_classic.chains import ConversationalRetrievalChain


In [76]:
qa = ConversationalRetrievalChain.from_llm(llm=model,retriever=vector_db.as_retriever(),condense_question_prompt=condense_question_prompt,return_source_documents = True,verbose=True)

## APP LOOP

In [ ]:
chain.invoke({

In [79]:
while True:
    chat_history = []
    print("=== DataBase Teacher ===")
    input_query = input("Hey I am Your Database Teacher \n Ask any thing what makes you doubt...\n Ask: ")
    if input_query.lower() == "exist":
        break
    result = qa({"question":input_query,"chat_history":chat_history})
    print(result['answer'])

=== DataBase Teacher ===


Hey I am Your Database Teacher 
 Ask any thing what makes you doubt...
 Ask:  what is database




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Oracle Database provides a centralized facility for the extraction, manipulation, and recreation of dictionary metadata. Oracle Database also supports all dictionary objects at their most current level.



Storage Manager

• Manages data storage on disks and memory.

• Uses indexing, partitioning, and file organization techniques.

Example: A B-tree index helps speed up search operations.





Storage engine in a database

The database storage engine is the core component of the DBMS that interacts with the file system at an OS level to store data. All SQL queries which interact with the underlying data go through the storage engine.

Which storage engine is the best for a database?

The right storage

Hey I am Your Database Teacher 
 Ask any thing what makes you doubt...
 Ask:  what is database give big high more large amount of answer




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Introduction to Database Systems

Definition, Basic Concepts, and Introduction

By:Dr.Subhasish Mohapatra

Text Book :Korth and Navathe



Agenda

Definition of a Database System

Importance of Database Systems

Basic Concepts

Database System Architecture

Advantages of Database Systems

Summary



Definition of Database System

Database: A structured collection of data stored electronically.

Database Management System (DBMS): Software that enables users to define, create, maintain, and control access to the database.

Key Characteristics:

 - Organized: Ensures data is structured systematically.

 - Persistent: Data remains available over time.

 - Accessible: Data can be retrieved and modified eas

Hey I am Your Database Teacher 
 Ask any thing what makes you doubt...
 Ask:  exist


In [5]:
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="mistral")
embedded_query = embeddings.embed_query("what is the full form of RAG ?")
print(f"Embedding length: {len(embedded_query)}")
print(embedded_query)

Embedding length: 4096
[3.1447765827178955, -6.816604137420654, 3.216777801513672, -0.711685299873352, -1.1622741222381592, -6.968747138977051, 3.392780065536499, 6.611316204071045, 6.993232250213623, -0.19267338514328003, -2.2250962257385254, -3.7824225425720215, -4.40477180480957, 5.219427108764648, -1.3858743906021118, -1.3037745952606201, 4.579098701477051, -3.418527841567993, 1.2805078029632568, 0.9897710680961609, 1.885888695716858, -4.384079933166504, -3.324239492416382, 3.4381747245788574, 5.25413179397583, -5.539207458496094, -2.630725383758545, -3.545428991317749, -6.406599521636963, 0.7449465990066528, 8.137307167053223, -2.736114025115967, -4.138625144958496, 2.457685947418213, 2.376807451248169, -7.882636070251465, -7.033960819244385, 3.9403584003448486, 0.09039206057786942, 5.305004596710205, 6.156126976013184, 5.201740264892578, 3.3413779735565186, -5.3687052726745605, -1.9183508157730103, 4.079753398895264, -0.4091106057167053, -4.417417049407959, -6.5270819664001465, 3

## To Illustrate How embeddings work,let's first generate the embeddings for two different sentences:

In [6]:
sentence1 = embeddings.embed_query("I love to watch youtube videos")
sentence2 = embeddings.embed_query("what is RAG ?")

In [7]:
#lets finding the similarity of sentence1 and sentence2 with embedded_query by help of cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
query_sentence1_similarity = cosine_similarity([embedded_query],[sentence1])[0][0]
print(f"Similarity of sentence1(query's vector) with embedded_query: {query_sentence1_similarity}")

#for 2 also
query_sentence2_similarity = cosine_similarity([embedded_query],[sentence2])[0][0]
print(f"Similarity of sentence2(query's vector) with embedded_query: {query_sentence2_similarity}")



Similarity of sentence1(query's vector) with embedded_query: 0.24995245127689425
Similarity of sentence2(query's vector) with embedded_query: 0.8784219006629612
